# Model & DB loading

In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from FlagEmbedding import BGEM3FlagModel

# load the model and processor
ckpt_siglip2 = "google/siglip2-giant-opt-patch16-384"
ckpt_bgem3 = "BAAI/bge-m3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor_siglip2 = AutoProcessor.from_pretrained(ckpt_siglip2)
model_siglip2 = AutoModel.from_pretrained(
    ckpt_siglip2,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
model_bgem3 = BGEM3FlagModel(
    ckpt_bgem3,
    normalize_embeddings=True,
    use_bf16=True,
    devices=device,
    return_dense=True,
    return_sparse=True,
    # return_colbert_vecs=True,
    model_kwargs={"attn_implementation": "flash_attention_2"}
)

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url='http://localhost:6333')

# Query encoding

In [38]:
QUERY = 'người đồng tính'
inputs = processor_siglip2(text=QUERY, padding='max_length', truncation=True, return_tensors='pt')

with torch.no_grad():
    query_dense_siglip2 = model_siglip2.get_text_features(**inputs.to(device)).pooler_output
    query_dense_siglip2 = F.normalize(query_dense_siglip2, dim=-1)

query_dense_siglip2 = query_dense_siglip2.to('cpu').squeeze().tolist()

query_all_bgem3 = model_bgem3.encode(QUERY)
query_dense_bgem3 = query_all_bgem3['dense_vecs'].squeeze().tolist()
query_sparse_bgem3 = models.SparseVector(
    indices=np.array(list(query_all_bgem3['lexical_weights'].keys()), dtype=np.int32).tolist(),
    values=np.array(list(query_all_bgem3['lexical_weights'].values()), dtype=np.float32).tolist(),
)

In [40]:
COLLECTION_NAME = 'test'
LIMIT = 100

prefetch = [
    models.Prefetch(
        query=query_dense_siglip2,
        using="dense_siglip2",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_dense_bgem3,
        using="dense_bgem3",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_sparse_bgem3,
        using='sparse_bgem3',
        limit=LIMIT * 2
    )
]

# Retrieval test

In [41]:
from IPython.display import display
import ipywidgets as widgets

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=prefetch,
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=LIMIT
).points

relevant_img_paths = [widgets.Image(value=open(hit.payload['img_path'], 'rb').read(), format='png', width=315) for hit in hits]
cols = 5
for i in range(0, len(relevant_img_paths), cols):
    if i + cols > len(relevant_img_paths):
        images = relevant_img_paths[i:len(relevant_img_paths)]
    else:
        images = relevant_img_paths[i:i+cols]

    row = widgets.HBox(images)
    display(row)